# FinRatioAnalysis MCP Server Example

This notebook demonstrates the **MCP (Model Context Protocol) Server** for FinRatioAnalysis.

## What is MCP?

MCP enables LLMs like Claude to invoke financial analysis tools through natural language. Instead of writing Python code, you can ask:
- *"What's Apple's ROIC trend over the last 4 years?"*
- *"Compare Microsoft and Google's leverage ratios"*
- *"Show me Tesla's complete financial snapshot"*

## What This Notebook Shows

Since MCP servers run as separate processes for LLM clients, this notebook **simulates tool invocations** to demonstrate:
1. All 11 MCP tools available
2. JSON and Markdown response formats
3. Error handling for invalid inputs
4. The aggregate `company_snapshot` tool

For actual LLM integration, see `specs/001-mcp-server/quickstart.md`.

## Setup: Import the MCP Server Module

We'll import the tools directly to simulate what an MCP client would invoke.

In [3]:
# Import MCP tools from individual modules
import sys
sys.path.append('..')  # Adjust if running from different directory

from finratioanalysis_mcp.tools.return_ratios import finratio_return_ratios
from finratioanalysis_mcp.tools.efficiency_ratios import finratio_efficiency_ratios
from finratioanalysis_mcp.tools.leverage_ratios import finratio_leverage_ratios
from finratioanalysis_mcp.tools.liquidity_ratios import finratio_liquidity_ratios
from finratioanalysis_mcp.tools.ccc import finratio_ccc
from finratioanalysis_mcp.tools.valuation_growth_metrics import finratio_valuation_growth_metrics
from finratioanalysis_mcp.tools.historical_valuation_metrics import finratio_historical_valuation_metrics
from finratioanalysis_mcp.tools.z_score import finratio_z_score
from finratioanalysis_mcp.tools.capm import finratio_capm
from finratioanalysis_mcp.tools.wacc import finratio_wacc
from finratioanalysis_mcp.tools.company_snapshot import finratio_company_snapshot

import json
import pandas as pd

## Tool 1: Return Ratios (Time Series)

Calculates profitability metrics: ROE, ROA, ROCE, ROIC, and margins.

In [4]:
# JSON format (default) - structured data for LLMs
result = finratio_return_ratios(
    ticker="AAPL",
    freq="yearly",
    response_format="json"
)

print("JSON Response:")
print(json.dumps(result, indent=2)[:500] + "...")  # First 500 chars

[04/17/26 16:47:33] INFO     req=a42ce663 Calling return_ratios for ticker=AAPL, freq=yearly           ]8;id=3926873;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926874;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:47:35] INFO     req=a42ce663 Retrieved return_ratios: shape=(5, 7)                       ]8;id=3926880;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926881;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

JSON Response:
{
  "data": [
    {
      "date": "2025-09-30",
      "ROE": 1.5191298333175105,
      "ROA": 0.3117962593356549,
      "ROCE": 0.6872062393471412,
      "ROIC": 0.8228358730470098,
      "GrossMargin": 0.4690516410716045,
      "OperatingMargin": 0.31970799762591884,
      "NetProfit": 0.2691506412181824
    },
    {
      "date": "2024-09-30",
      "ROE": 1.6459350307287095,
      "ROA": 0.25682503150857583,
      "ROCE": 0.6533607652660827,
      "ROIC": 0.6998997671891681,
      "GrossMargi...


In [6]:
# Markdown format - human-readable tables
result_md = finratio_return_ratios(
    ticker="AAPL",
    freq="yearly",
    response_format="markdown"
)

print("Markdown Response:")
print(result_md['data'][:600])  # First 600 chars

[04/17/26 16:47:56] INFO     req=d313aa9d Calling return_ratios for ticker=AAPL, freq=yearly           ]8;id=3926898;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926899;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:47:57] INFO     req=d313aa9d Retrieved return_ratios: shape=(5, 7)                       ]8;id=3926904;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926905;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

Markdown Response:
| date | ROE | ROA | ROCE | ROIC | GrossMargin | OperatingMargin | NetProfit |
| ---- | --- | --- | ---- | ---- | ----------- | --------------- | --------- |
| 2025-09-30 | 1.51913 | 0.311796 | 0.687206 | 0.822836 | 0.469052 | 0.319708 | 0.269151 |
| 2024-09-30 | 1.64594 | 0.256825 | 0.653361 | 0.6999 | 0.462063 | 0.315102 | 0.239713 |
| 2023-09-30 | 1.56076 | 0.275098 | 0.551446 | 0.680376 | 0.441311 | 0.298214 | 0.253062 |
| 2022-09-30 | 1.96959 | 0.282924 | 0.600871 | 0.627455 | 0.433096 | 0.302887 | 0.253096 |
| 2021-09-30 | null | null | null | null | null | null | null |


## Tool 2: Efficiency Ratios

Includes **FCF Margin** and **Reinvestment Rate** (VCG Quality Metrics).

In [8]:
result = finratio_efficiency_ratios(
    ticker="MSFT",
    freq="yearly",
    response_format="json"
)

periods = result['data']
print(f"Symbol: MSFT")
print(f"Number of periods: {len(periods)}")
print(f"\nLatest period metrics:")
latest = periods[0]
print(f"  Date: {latest['date']}")
print(f"  FCF_Margin: {latest.get('FCF_Margin', 'N/A')}")
print(f"  ReinvestmentRate: {latest.get('ReinvestmentRate', 'N/A')}")

[04/17/26 16:49:41] INFO     req=e4c49f90 Calling efficiency_ratios for ticker=MSFT, freq=yearly       ]8;id=3926922;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926923;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:49:43] INFO     req=e4c49f90 Retrieved efficiency_ratios: shape=(5, 6)                   ]8;id=3926928;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926929;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

Symbol: MSFT
Number of periods: 5

Latest period metrics:
  Date: 2021-06-30
  FCF_Margin: None
  ReinvestmentRate: None


## Tool 3: Leverage Ratios

Debt-to-Equity, Equity Ratio, Debt Ratio.

In [9]:
result = finratio_leverage_ratios(
    ticker="GOOGL",
    freq="yearly",
    response_format="json"
)

# Extract key metrics
periods = result['data']
print(f"Symbol: GOOGL")
print(f"Latest Debt-to-Equity Ratio: {periods[0].get('Debt_Equity_Ratio', 'N/A')}")

[04/17/26 16:49:49] INFO     req=6a143044 Calling leverage_ratios for ticker=GOOGL, freq=yearly        ]8;id=3926934;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926935;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:49:51] INFO     req=6a143044 Retrieved leverage_ratios: shape=(5, 3)                     ]8;id=3926940;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926941;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

Symbol: GOOGL
Latest Debt-to-Equity Ratio: N/A


## Tool 4: Liquidity Ratios

Current, Quick, and Cash Ratios for short-term solvency analysis.

In [10]:
result = finratio_liquidity_ratios(
    ticker="TSLA",
    freq="yearly",
    response_format="json"
)

periods = result['data']
print(f"Symbol: TSLA")
print(f"Number of periods: {len(periods)}")
print(f"\nLatest liquidity metrics:")
latest = periods[0]
print(f"  Current Ratio: {latest.get('CurrentRatio', 'N/A')}")
print(f"  Quick Ratio: {latest.get('QuickRatio', 'N/A')}")

[04/17/26 16:49:56] INFO     req=dc51a4a4 Calling liquidity_ratios for ticker=TSLA, freq=yearly        ]8;id=3926946;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926947;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:49:57] INFO     req=dc51a4a4 Retrieved liquidity_ratios: shape=(5, 8)                    ]8;id=3926952;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926953;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

Symbol: TSLA
Number of periods: 5

Latest liquidity metrics:
  Current Ratio: N/A
  Quick Ratio: 0.6649744592293624


## Tool 5: Cash Conversion Cycle (CCC)

Measures how quickly a company converts inventory to cash.

In [11]:
result = finratio_ccc(
    ticker="NVDA",
    freq="yearly",
    response_format="json"
)

periods = result['data']
print(f"Symbol: NVDA")
print(f"\nLatest CCC:")
latest = periods[0]
print(f"  Date: {latest['date']}")
print(f"  CCC (days): {latest.get('CCC', 'N/A')}")
print(f"  DIO (Days Inventory Outstanding): {latest.get('DIO', 'N/A')}")
print(f"  DSO (Days Sales Outstanding): {latest.get('DSO', 'N/A')}")

[04/17/26 16:50:03] INFO     req=cdb3eca8 Calling ccc for ticker=NVDA, freq=yearly                     ]8;id=3926958;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926959;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:50:05] INFO     req=cdb3eca8 Retrieved ccc: shape=(5, 4)                                 ]8;id=3926964;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926965;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

Symbol: NVDA

Latest CCC:
  Date: 2026-01-31
  CCC (days): 63.84538471308277
  DIO (Days Inventory Outstanding): 61.23353341336534
  DSO (Days Sales Outstanding): 31.84398415285869


## Tool 6: Valuation & Growth Metrics (Snapshot)

Single-row snapshot with current market values and 3-year CAGR.

In [12]:
result = finratio_valuation_growth_metrics(
    ticker="AAPL",
    freq="yearly",
    response_format="json"
)

data = result['data']
print(f"Symbol: AAPL")
print(f"\nCurrent Valuation:")
print(f"  P/E Ratio: {data.get('pe_ratio', 'N/A')}")
print(f"  EV/EBITDA: {data.get('ev_ebitda', 'N/A')}")
print(f"  FCF Yield: {data.get('fcf_yield', 'N/A')}")
print(f"\nGrowth (3Y CAGR):")
print(f"  Revenue: {data.get('revenue_cagr_3y', 'N/A')}")
print(f"  Net Income: {data.get('net_income_cagr_3y', 'N/A')}")

[04/17/26 16:50:11] INFO     req=73386019 Calling valuation_growth_metrics for ticker=AAPL,            ]8;id=3926970;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926971;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\
                             freq=yearly                                                                           

[04/17/26 16:50:12] INFO     req=73386019 Retrieved valuation_growth_metrics: shape=(1, 10)           ]8;id=3926976;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926977;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

Symbol: AAPL

Current Valuation:
  P/E Ratio: 34.20633
  EV/EBITDA: 27.872879433166606
  FCF Yield: 0.02486693411803101

Growth (3Y CAGR):
  Revenue: 0.01812535743479393
  Net Income: 0.039212592033254


## Tool 7: Historical Valuation Metrics (Time Series)

**NEW**: P/E Ratio, EV/EBITDA, and FCF Yield over multiple periods.

In [13]:
result = finratio_historical_valuation_metrics(
    ticker="AAPL",
    freq="yearly",
    response_format="json"
)

periods = result['data']
print(f"Symbol: AAPL")
print(f"Number of periods: {len(periods)}")
print(f"\nHistorical P/E Ratios (last 3 years):")
for period in periods[:3]:
    print(f"  {period['date']}: P/E = {period.get('P/E', 'N/A')}")

[04/17/26 16:50:17] INFO     req=b553d67d Calling historical_valuation_metrics for ticker=AAPL,        ]8;id=3926982;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926983;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\
                             freq=yearly                                                                           

[04/17/26 16:50:18] INFO     req=b553d67d Retrieved historical_valuation_metrics: shape=(5, 4)        ]8;id=3926988;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926989;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

Symbol: AAPL
Number of periods: 5

Historical P/E Ratios (last 3 years):
  2025-09-30: P/E = N/A
  2024-09-30: P/E = N/A
  2023-09-30: P/E = N/A


## Tool 8: Altman Z-Score

Bankruptcy prediction metric (snapshot).

In [14]:
result = finratio_z_score(
    ticker="AAPL",
    freq="yearly",
    response_format="json"
)

data = result['data']
print(f"Symbol: AAPL")
print(f"Z-Score: {data.get('Z_Score', 'N/A')}")
print(f"Zone: {data.get('Zone', 'N/A')}")
print(f"\nInterpretation:")
print(f"  Z > 2.99: Safe Zone (low bankruptcy risk)")
print(f"  1.81 < Z < 2.99: Grey Zone")
print(f"  Z < 1.81: Distress Zone (high bankruptcy risk)")

[04/17/26 16:50:23] INFO     req=5d8b6a0a Calling z_score for ticker=AAPL, freq=yearly                 ]8;id=3926994;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3926995;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:50:25] INFO     req=5d8b6a0a Retrieved z_score: shape=(1, 3)                             ]8;id=3927000;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927001;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

Symbol: AAPL
Z-Score: N/A
Zone: Safe Zone

Interpretation:
  Z > 2.99: Safe Zone (low bankruptcy risk)
  1.81 < Z < 2.99: Grey Zone
  Z < 1.81: Distress Zone (high bankruptcy risk)


## Tool 9: CAPM (Capital Asset Pricing Model)

Expected return and Sharpe Ratio (snapshot).

In [15]:
result = finratio_capm(
    ticker="MSFT",
    freq="yearly",
    response_format="json"
)

data = result['data']
print(f"Symbol: MSFT")
print(f"Expected Return (CAPM): {data.get('CAPM', 'N/A')}")
print(f"Sharpe Ratio: {data.get('Sharpe', 'N/A')}")

[04/17/26 16:50:30] INFO     req=9ece095d Calling capm for ticker=MSFT, freq=yearly                    ]8;id=3927006;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927007;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:50:31] INFO     req=9ece095d Retrieved capm: shape=(1, 3)                                ]8;id=3927012;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927013;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

Symbol: MSFT
Expected Return (CAPM): 0.1472
Sharpe Ratio: 0.362


## Tool 10: WACC (Weighted Average Cost of Capital)

The company's blended cost of debt and equity financing (snapshot).

In [20]:
result = finratio_wacc(
    ticker="GOOGL",
    freq="yearly",
    response_format="json"
)

data = result['data']
print(f"Symbol: GOOGL")
print(f"WACC: {data.get('WACC', 'N/A')}")

[04/17/26 16:52:08] INFO     req=5a1d994a Calling wacc for ticker=GOOGL, freq=yearly                   ]8;id=3927390;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927391;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:52:09] INFO     req=5a1d994a Retrieved wacc: shape=(1, 2)                                ]8;id=3927396;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927397;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

Symbol: GOOGL
WACC: 0.14463121869813753


## Tool 11: Company Snapshot (Aggregate)

**The Power Tool**: Combines all 10 methods into a single comprehensive report.

This is what makes MCP powerful - an LLM can ask "Tell me everything about Apple" and get a complete financial profile in one call.

In [18]:
# Get complete snapshot for Apple (yearly data)
result = finratio_company_snapshot(
    ticker="AAPL",
    freq="yearly",
    response_format="json"
)

print(f"Tool: company_snapshot")
print(f"Symbol: {result.get('ticker', 'AAPL')}")
print(f"Frequency: {result.get('freq', 'yearly')}")
print(f"\nAvailable sections:")

sections = result.get('sections', {})
for section_name, section_data in sections.items():
    status = section_data.get('status', 'unknown')
    if status == 'ok':
        data = section_data.get('data')
        if isinstance(data, list):
            print(f"  ✓ {section_name}: {len(data)} periods")
        else:
            print(f"  ✓ {section_name}: snapshot data")
    else:
        error = section_data.get('error', {})
        print(f"  ✗ {section_name}: {error.get('code', 'ERROR')}")

[04/17/26 16:51:12] INFO     req=a3b242f9 Calling return_ratios for ticker=AAPL, freq=yearly           ]8;id=3927150;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927151;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:51:13] INFO     req=a3b242f9 Retrieved return_ratios: shape=(5, 7)                       ]8;id=3927156;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927157;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

                    INFO     req=6a074ad3 Calling efficiency_ratios for ticker=AAPL, freq=yearly       ]8;id=3927162;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927163;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:51:14] INFO     req=6a074ad3 Retrieved efficiency_ratios: shape=(5, 6)                   ]8;id=3927168;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927169;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

                    INFO     req=f6bab4d0 Calling leverage_ratios for ticker=AAPL, freq=yearly         ]8;id=3927174;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927175;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

...

                    INFO     req=e65f4672 Calling capm for ticker=AAPL, freq=yearly                    ]8;id=3927246;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927247;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:51:21] INFO     req=e65f4672 Retrieved capm: shape=(1, 3)                                ]8;id=3927252;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927253;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

                    INFO     req=e82ef123 Calling wacc for ticker=AAPL, freq=yearly                    ]8;id=3927258;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927259;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:51:22] INFO     req=e82ef123 Retrieved wacc: shape=(1, 2)                                ]8;id=3927264;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927265;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

Tool: company_snapshot
Symbol: AAPL
Frequency: yearly

Available sections:
  ✓ return_ratios: 5 periods
  ✓ efficiency_ratios: 5 periods
  ✓ leverage_ratios: 5 periods
  ✓ liquidity_ratios: 5 periods
  ✓ ccc: 5 periods
  ✓ historical_valuation_metrics: 5 periods
  ✓ valuation_growth_metrics: snapshot data
  ✓ z_score: snapshot data
  ✓ capm: snapshot data
  ✓ wacc: snapshot data


In [23]:
# Markdown format provides a beautiful, human-readable report
result_md = finratio_company_snapshot(
    ticker="AAPL",
    freq="yearly",
    response_format="markdown"
)

# For company_snapshot, markdown returns JSON structure (could be enhanced for formatted reports)
print(f"Company snapshot (markdown requested): {len(result_md.get('sections', {}))} sections available")
print(f"\nExample section (return_ratios):")
ret_data = result_md.get('sections', {}).get('return_ratios', {}).get('data', [])
if ret_data and isinstance(ret_data, list):
    print(f"  First period: {ret_data[0]}")

[04/17/26 16:53:00] INFO     req=c0ca527a Calling return_ratios for ticker=AAPL, freq=yearly           ]8;id=3927522;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927523;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:53:01] INFO     req=c0ca527a Retrieved return_ratios: shape=(5, 7)                       ]8;id=3927528;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927529;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

                    INFO     req=d07b7258 Calling efficiency_ratios for ticker=AAPL, freq=yearly       ]8;id=3927534;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927535;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:53:02] INFO     req=d07b7258 Retrieved efficiency_ratios: shape=(5, 6)                   ]8;id=3927540;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927541;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

                    INFO     req=8db21dc9 Calling leverage_ratios for ticker=AAPL, freq=yearly         ]8;id=3927546;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927547;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

...

                    INFO     req=8bde4e24 Calling capm for ticker=AAPL, freq=yearly                    ]8;id=3927618;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927619;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:53:10] INFO     req=8bde4e24 Retrieved capm: shape=(1, 3)                                ]8;id=3927624;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927625;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

                    INFO     req=3c03c7af Calling wacc for ticker=AAPL, freq=yearly                    ]8;id=3927630;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927631;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:53:11] INFO     req=3c03c7af Retrieved wacc: shape=(1, 2)                                ]8;id=3927636;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927637;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

Company snapshot (markdown requested): 10 sections available

Example section (return_ratios):
  First period: {'date': '2025-09-30', 'ROE': 1.5191298333175105, 'ROA': 0.3117962593356549, 'ROCE': 0.6872062393471412, 'ROIC': 0.8228358730470098, 'GrossMargin': 0.4690516410716045, 'OperatingMargin': 0.31970799762591884, 'NetProfit': 0.2691506412181824}


## Error Handling: Invalid Ticker

MCP tools return structured error responses (no stack traces to clients).

In [22]:
# Try an invalid ticker
result = finratio_return_ratios(
    ticker="INVALID_TICKER_XYZ",
    freq="yearly",
    response_format="json"
)

print("Error response:")
print(json.dumps(result, indent=2))

Error response:
{
  "code": "INVALID_TICKER",
  "message": "Invalid ticker format. Must be upper-case alphanumerics with optional '.' or '-' (e.g., AAPL, BRK-B).",
  "details": {
    "field": "ticker",
    "pydantic_error": "String should have at most 10 characters"
  }
}


## Error Handling: Unsupported Frequency

Only "yearly" and "quarterly" are supported.

In [24]:
# Try invalid frequency
result = finratio_efficiency_ratios(
    ticker="AAPL",
    freq="monthly",  # Invalid - only 'yearly' or 'quarterly'
    response_format="json"
)

print("Error response:")
print(json.dumps(result, indent=2))

Error response:
{
  "code": "UNSUPPORTED_FREQ",
  "message": "Unsupported frequency. Must be 'yearly' or 'quarterly'.",
  "details": {
    "field": "freq",
    "pydantic_error": "Input should be 'yearly' or 'quarterly'"
  }
}


## Quarterly Data Example

Most tools support quarterly frequency for more granular analysis.

In [25]:
# Get quarterly data (more periods)
result = finratio_return_ratios(
    ticker="AAPL",
    freq="quarterly",
    response_format="json"
)

periods = result['data']
print(f"Symbol: AAPL")
print(f"Frequency: quarterly")
print(f"Number of periods: {len(periods)}")
print(f"\nLast 3 quarters:")
for period in periods[:3]:
    print(f"  {period['date']}: ROE={period.get('ROE', 'N/A')}, ROIC={period.get('ROIC', 'N/A')}")

[04/17/26 16:53:29] INFO     req=550025c8 Calling return_ratios for ticker=AAPL, freq=quarterly        ]8;id=3927642;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927643;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:53:31] INFO     req=550025c8 Retrieved return_ratios: shape=(6, 7)                       ]8;id=3927648;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927649;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

Symbol: AAPL
Frequency: quarterly
Number of periods: 6

Last 3 quarters:
  2024-09-30: ROE=None, ROIC=None
  2024-12-31: ROE=0.5442044399173133, ROIC=0.2742167679050142
  2025-03-31: ROE=0.3709802982214504, ROIC=0.18283788351197558


## Comparing Multiple Companies

Practical example: Compare tech giants across key metrics.

In [26]:
# Compare FAANG stocks on valuation metrics
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META']

print("Tech Giants Valuation Comparison:\n")
for ticker in tickers:
    result = finratio_valuation_growth_metrics(
        ticker=ticker,
        freq="yearly",
        response_format="json"
    )
    
    if 'error' not in result:
        data = result['data']
        print(f"{ticker}:")
        print(f"  P/E: {data.get('pe_ratio', 'N/A')}")
        print(f"  EV/EBITDA: {data.get('ev_ebitda', 'N/A')}")
        print(f"  Revenue CAGR (3Y): {data.get('revenue_cagr_3y', 'N/A')}")
        print()

Tech Giants Valuation Comparison:



                    INFO     req=766b096f Calling valuation_growth_metrics for ticker=AAPL,            ]8;id=3927654;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927655;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\
                             freq=yearly                                                                           

[04/17/26 16:53:33] INFO     req=766b096f Retrieved valuation_growth_metrics: shape=(1, 10)           ]8;id=3927660;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927661;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

AAPL:
  P/E: 34.20633
  EV/EBITDA: 27.872879433166606
  Revenue CAGR (3Y): 0.01812535743479393



                    INFO     req=e8424d41 Calling valuation_growth_metrics for ticker=MSFT,            ]8;id=3927666;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927667;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\
                             freq=yearly                                                                           

...

[04/17/26 16:53:37] INFO     req=8038ea32 Retrieved valuation_growth_metrics: shape=(1, 10)           ]8;id=3927696;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927697;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

AMZN:
  P/E: 34.945606
  EV/EBITDA: 16.697384519798476
  Revenue CAGR (3Y): 0.11731283601806397



                    INFO     req=5833dd56 Calling valuation_growth_metrics for ticker=META,            ]8;id=3927702;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927703;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\
                             freq=yearly                                                                           

[04/17/26 16:53:39] INFO     req=5833dd56 Retrieved valuation_growth_metrics: shape=(1, 10)           ]8;id=3927708;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927709;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

META:
  P/E: 29.312473
  EV/EBITDA: 16.93026750087501
  Revenue CAGR (3Y): 0.19893831442939258



## Quality Analysis: ROIC Trends

Track Return on Invested Capital over time for capital efficiency.

In [27]:
# Analyze ROIC trends for Apple
result = finratio_return_ratios(
    ticker="AAPL",
    freq="yearly",
    response_format="json"
)

periods = result['data']
print("Apple ROIC Trends (Last 5 Years):\n")
for period in periods[:5]:
    roic = period.get('ROIC', 'N/A')
    print(f"{period['date']}: ROIC = {roic}")

                    INFO     req=0b5f8e76 Calling return_ratios for ticker=AAPL, freq=yearly           ]8;id=3927714;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927715;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:53:40] INFO     req=0b5f8e76 Retrieved return_ratios: shape=(5, 7)                       ]8;id=3927720;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927721;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#100\100]8;;\

Apple ROIC Trends (Last 5 Years):

2025-09-30: ROIC = 0.8228358730470098
2024-09-30: ROIC = 0.6998997671891681
2023-09-30: ROIC = 0.6803763316950044
2022-09-30: ROIC = 0.6274552499465476
2021-09-30: ROIC = None


## Banking Sector: Handling Missing Fields

Banks don't have inventory or gross profit. MCP tools gracefully handle missing data.

In [ ]:
# Analyze a bank (JPMorgan Chase)
result = finratio_efficiency_ratios(
    ticker="JPM",
    freq="yearly",
    response_format="json"
)

# Check if result is error response (has 'code' key) or success response (has 'data' key)
if 'code' in result:
    # Error response (top-level error structure)
    print(f"Error: {result.get('message', 'Unknown error')}")
    print(f"Code: {result.get('code', 'N/A')}")
elif 'data' in result:
    # Success response
    periods = result['data']
    print(f"Symbol: JPM")
    print(f"\nBank efficiency metrics:")
    latest = periods[0]
    for key, value in latest.items():
        if key != 'date':
            print(f"  {key}: {value}")
            
    print("\nNote: Some fields may be null for banks (e.g., Inventory Turnover)")
else:
    print(f"Unexpected result format: {result}")

[04/17/26 16:54:04] INFO     req=40f17b55 Calling efficiency_ratios for ticker=JPM, freq=yearly        ]8;id=3927751;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927752;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#57\57]8;;\

[04/17/26 16:54:06] ERROR    req=40f17b55 Unexpected error in _call_library                           ]8;id=3927757;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py\server.py]8;;\:]8;id=3927758;file://c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysis_mcp\server.py#132\132]8;;\
                             ╭───────────────── Traceback (most recent call last) ──────────────────╮              
                             │ c:\Users\loren\Documents\FinRatioAnalysis\.venv\Lib\site-packages\pa │              
                             │ ndas\core\indexes\base.py:3641 in get_loc                            │              
                             │                                                                      │              
                             │   3638 │   │   """                                                   │              
                             │   3639 │   │   casted_key = self._maybe_cast_indexer(key)            │              
                             │   3640 │   │   try:                                                  │              
                             │ ❱ 3641 │   │   │   return self._engine.get_loc(casted_key)           │              
                             │   3642 │   │   except KeyError as err:                               │              
                             │   3643 │   │   │   if isinstance(casted_key, slice) or (             │              
                             │   3644 │   │   │   │   isinstance(casted_key, abc.Iterable)          │              
                             │                                                                      │              
                             │ in pandas._libs.index.IndexEngine.get_loc:168                        │              
                             │                                                                      │              
                             │ in pandas._libs.index.IndexEngine.get_loc:197                        │              
                             │                                                                      │              
                             │ in pandas._libs.hashtable.PyObjectHashTable.get_item:7668            │              
                             │                                                                      │              
                             │ in pandas._libs.hashtable.PyObjectHashTable.get_item:7676            │              
                             ╰──────────────────────────────────────────────────────────────────────╯              
                             KeyError: 'CostOfRevenue'                                                             
                                                                                                                   
                             The above exception was the direct cause of the following exception:                  
                                                                                                                   
                             ╭───────────────── Traceback (most recent call last) ──────────────────╮              
                             │ c:\Users\loren\Documents\FinRatioAnalysis\example\..\finratioanalysi │              
                             │ s_mcp\server.py:71 in _call_library                                  │              
                             │                                                                      │              
                             │    68 │   │   │   │   details={"method": method_name, "req_id": req_ │              
                             │    69 │   │   │   )                                                  │              
                             │    70 │   │                                                          │              
                             │ ❱  71 │   │   result = ge

Error: An unexpected error occurred while fetching financial data. (ref=40f17b55)
Code: INTERNAL_ERROR


: 

## Summary: MCP vs Direct Python Usage

### Direct Python (Traditional)
```python
from FinRatioAnalysis import FinRatioAnalysis
analyzer = FinRatioAnalysis('AAPL', 'yearly')
df = analyzer.return_ratios()
```

### MCP Server (LLM Integration)
```
User: "What's Apple's ROIC for the last 4 years?"
Claude: [invokes finratio_return_ratios tool]
        [returns structured data]
        "Apple's ROIC over the last 4 years shows..."
```

## Key Benefits:
1. **Natural Language Interface** - No code required
2. **Structured Output** - JSON/Markdown formats optimized for LLMs
3. **Error Handling** - Clean error messages without stack traces
4. **Aggregate Tools** - Single-call comprehensive reports
5. **Read-Only & Open-World** - Safe for LLM automation

## Next Steps

1. **Setup Claude Desktop**: See `specs/001-mcp-server/quickstart.md`
2. **Run MCP Inspector**: Test tools interactively
3. **Try Natural Language Queries**: Ask Claude financial questions
4. **Build Custom Workflows**: Chain multiple tool calls for complex analysis

---

For more details:
- MCP Quickstart: `specs/001-mcp-server/quickstart.md`
- Tool Contracts: `specs/001-mcp-server/contracts/*.json`
- Test Suite: `tests_mcp/`